In [ ]:
# ===================================================================
# SISTEMA DE ANÁLISIS AMBIENTAL PARA AULA 0.13
# ===================================================================
# AUTOR: Carmen Gómez Alarcón
# FECHA: 2026
# FASE: Paso 4. Selección de características y normalización
# DESCRIPCIÓN:
#   Este módulo realiza la selección de características relevantes
#   (procedentes de step 2) y la normalización de los datos
#   (StandardScaler) para su posterior uso en modelos de regresión.
#   Solo para TRAINING DATA set de caracteristicas (x)
# ===================================================================

import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler  # Para normalizar datos (media=0, std=1)
from sklearn.feature_selection import SelectKBest, mutual_info_regression  # Para seleccionar mejores features
import pickle  # Para guardar/cargar objetos de Python
import warnings
warnings.filterwarnings('ignore')  # Ocultar warnings no críticos

# Directorios de entrada y salida
INPUT_DIR  = 'ml_features_grouped'  # Directorio con datos pre-procesados
OUTPUT_DIR = 'ml_normalized_grouped'  # Directorio para guardar resultados
os.makedirs(OUTPUT_DIR, exist_ok=True)  # Crear directorio si no existe

# ===================================================================
# 1. CARGA DE DATOS DE ENTRENAMIENTO Y PRUEBA
# ===================================================================
X_train = pd.read_excel(os.path.join(INPUT_DIR, 'X_train.xlsx')).values
X_test  = pd.read_excel(os.path.join(INPUT_DIR, 'X_test.xlsx')).values
y_train = pd.read_excel(os.path.join(INPUT_DIR, 'y_train_regression.xlsx')).values.ravel()
y_test  = pd.read_excel(os.path.join(INPUT_DIR, 'y_test_regression.xlsx')).values.ravel()

# Cargar nombres de las características (ej: 'temperatura', 'co2', etc.)
with open(os.path.join(INPUT_DIR, 'feature_names.pkl'), 'rb') as f:
    feature_names = pickle.load(f)

# Mostrar dimensiones de los datasets
print(f"   X_train: {X_train.shape}, y_train: {y_train.shape}")  # (filas, columnas)
print(f"   X_test:  {X_test.shape},  y_test:  {y_test.shape}")

# Vista previa de los datos (primeras 3 filas, primeras 5 columnas)
print("\n   Vista previa X_train (primeras 3 filas, 5 features):")
print(pd.DataFrame(X_train, columns=feature_names).iloc[:3, :5].to_string())


# ===================================================================
# 2. SELECCIÓN DE CARACTERÍSTICAS (AJUSTE EXCLUSIVO EN TRAIN)
# ===================================================================
# Objetivo: Quedarse solo con las variables que realmente ayudan a predecir ocupación
# Esto reduce ruido, mejora rendimiento y evita sobreajuste

# Cálculo de información mutua para evaluar relevancia de cada feature
# Información mutua = cuánto conocimiento aporta X sobre y (ocupación)
print("\n   Scores per feature (train only):")

# Calcular score de información mutua para cada característica
scores = mutual_info_regression(X_train, y_train, random_state=42)  # random_state para reproducibilidad
# Ordenar features por score (de mayor a menor importancia)
feature_scores = sorted(zip(feature_names, scores), key=lambda x: x[1], reverse=True)

# Mostrar top 20 características más relevantes
for name, score in feature_scores[:20]:
    print(f"      {name:<50s} {score:.4f}")

# Selección de features con score positivo (>0.01) limitando a máximo 10
# k = número de features a seleccionar
k = sum(s > 0.01 for _, s in feature_scores)  # Contar cuántas tienen score > 0.01
k = min(k, 10)  # Limitar a máximo 10 features para evitar sobreajuste
print(f"\n   Selecting k={k} features (score > 0, max 10)")

# Crear selector que elige las k mejores features según información mutua
selector = SelectKBest(mutual_info_regression, k=k)
X_train_sel = selector.fit_transform(X_train, y_train)  # Ajustar y transformar entrenamiento
X_test_sel  = selector.transform(X_test)  # Solo transformar prueba (mismo selector)

# Obtener nombres de las características seleccionadas
selected_features = np.array(feature_names)[selector.get_support()].tolist()
print(f"\n   Selected {len(selected_features)} features:")
for i, feat in enumerate(selected_features):
    print(f"      {i+1:2d}. {feat}")

# Vista previa después de selección
print("\n   Vista previa tras selección (primeras 3 filas):")
print(pd.DataFrame(X_train_sel, columns=selected_features).iloc[:3].to_string())


# ==========================================
# 3. NORMALIZAR (fit SOLO en train)
# ==========================================
# Objetivo: Escalar datos para que tengan media=0 y desviación típica=1
# Importante: Los modelos lineales y SVR necesitan datos normalizados
# Árboles de decisión NO necesitan normalización (por eso guardamos ambas versiones)

print("\n3. NORMALIZING...")

# Crear escalador StandardScaler (media=0, std=1)
scaler = StandardScaler()
X_train_norm = scaler.fit_transform(X_train_sel)  # fit: calcula media y std del TRAIN + transforma
X_test_norm  = scaler.transform(X_test_sel)        # transform: aplica media/std del TRAIN al TEST

# Verificar que la normalización funcionó correctamente
print(f"   Train → mean: {X_train_norm.mean():.4f}, std: {X_train_norm.std():.4f}")  # Debe ser ~0 y ~1
print(f"   Test  → mean: {X_test_norm.mean():.4f},  std: {X_test_norm.std():.4f}")   # No exacto pero cercano

# Vista previa después de normalización
print("\n   Vista previa tras normalización (primeras 3 filas):")
print(pd.DataFrame(X_train_norm, columns=selected_features).iloc[:3].to_string())

# Comparar una característica antes/después de normalizar
print("\n   Comparación de una feature antes/después:")
feat = selected_features[0]  # Primera característica seleccionada
idx = feature_names.index(feat)  # Encontrar su índice original
print(f"   {feat}:")
print(f"      Antes → min: {X_train[:,idx].min():.2f}, max: {X_train[:,idx].max():.2f}, mean: {X_train[:,idx].mean():.2f}")
print(f"      Después → min: {X_train_norm[:,0].min():.2f}, max: {X_train_norm[:,0].max():.2f}, mean: {X_train_norm[:,0].mean():.2f}")

# ==========================================
# 4. GUARDAR
# ==========================================
print("\n4. SAVING...")

# Versión SIN normalizar (para modelos basados en árboles: Random Forest, XGBoost, etc.)
# Estos modelos no necesitan normalización porque trabajan con rangos/divisiones
pd.DataFrame(X_train_sel, columns=selected_features).to_excel(
    os.path.join(OUTPUT_DIR, 'X_train.xlsx'), index=False)
pd.DataFrame(X_test_sel, columns=selected_features).to_excel(
    os.path.join(OUTPUT_DIR, 'X_test.xlsx'), index=False)
pd.DataFrame({'Occupancy': y_train}).to_excel(
    os.path.join(OUTPUT_DIR, 'y_train.xlsx'), index=False)
pd.DataFrame({'Occupancy': y_test}).to_excel(
    os.path.join(OUTPUT_DIR, 'y_test.xlsx'), index=False)

# Versión NORMALIZADOS (para modelos lineales: Regresión Lineal, SVR, Redes Neuronales)
# Estos modelos requieren que todas las variables estén en la misma escala
pd.DataFrame(X_train_norm, columns=selected_features).to_excel(
    os.path.join(OUTPUT_DIR, 'X_train_normalised.xlsx'), index=False)
pd.DataFrame(X_test_norm, columns=selected_features).to_excel(
    os.path.join(OUTPUT_DIR, 'X_test_normalised.xlsx'), index=False)

# Guardar objetos para reproducibilidad (poder aplicar mismo pre-procesado a nuevos datos)
with open(os.path.join(OUTPUT_DIR, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)  # Guardar escalador (medias y desviaciones)
with open(os.path.join(OUTPUT_DIR, 'selector.pkl'), 'wb') as f:
    pickle.dump(selector, f)  # Guardar selector (qué features se eligieron)
with open(os.path.join(OUTPUT_DIR, 'selected_features.pkl'), 'wb') as f:
    pickle.dump(selected_features, f)  # Guardar lista de features seleccionadas

# ==========================================
# RESUMEN FINAL
# ==========================================
print("\n" + "="*60)
print("DONE!")
print("="*60)
print(f"   Features originales: {len(feature_names)}")  # Cuántas habíamos
print(f"   Features seleccionadas: {len(selected_features)}")  # Cuántas nos quedamos
print(f"   X_train final: {X_train_norm.shape}")  # Dimensiones finales
print(f"   X_test  final: {X_test_norm.shape}")
print(f"\n   Saved in {OUTPUT_DIR}/:")
# Mostrar todos los archivos guardados con su tamaño
for fname in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, fname)) / 1024
    print(f"      {fname} ({size:.0f} KB)")
print("="*60)

   X_train: (58, 96), y_train: (58,)
   X_test:  (25, 96),  y_test:  (25,)

   Vista previa X_train (primeras 3 filas, 5 features):
   Corridor_CO2_Increment_max  Corridor_CO2_Increment_mean  Corridor_CO2_Increment_min  Corridor_CO2_Increment_std  Corridor_CO2_max
0                       401.0                    11.321429                      -210.0                   72.444340            1471.0
1                       283.0                    38.357143                         0.0                   94.115132            2008.0
2                        86.0                   -16.139535                      -547.0                   74.280450            2008.0

   Scores per feature (train only):
      Corridor_CO2_min                                   0.7149
      Corridor_CO2_mean                                  0.6984
      Lighting_Consumption_Increment_min                 0.6853
      Hanging_CO2_min                                    0.6199
      Corridor_Temp_Increment_mean         